# Count based aggregation

In [ ]:
import sys
ROOT = "/workspaces/Object-Centric-Concept-Bottlenecks"
sys.path.insert(0, ROOT)

import os
import glob
import json
import pickle
import h5py
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from tqdm import tqdm

from eval_utils import PATHS, METHODS, NUM_OBJECTS, VOC_CLASSES, VOC_SYNONYMS, vocab_idx

PATHS = PATHS["voc"]

# filtering as in generate_concepts.py
MIN_OBJ_SIZE   = 0.01
MAX_OBJ_SIZE   = 0.85
IOU_THRESHOLD  = 0.5
PROPOSAL_SCORE = 0.5   
MIN_SCORE      = 0.2   

DET_CACHE = f"{ROOT}/red_circle_notebooks/count_aggregation_det_cache_voc.pkl"

COCO_TO_VOC = {
    "airplane": "aeroplane", "bicycle": "bicycle", "bird": "bird", "boat": "boat", "bottle": "bottle",
    "bus": "bus", "car": "car", "cat": "cat", "chair": "chair", "cow": "cow",
    "dining table": "diningtable", "dog": "dog", "horse": "horse", "motorcycle": "motorbike",
    "person": "person", "potted plant": "pottedplant", "sheep": "sheep", "couch": "sofa",
    "train": "train", "tv": "tvmonitor",
}
print({k: os.path.exists(p) for k, p in PATHS.items()})

In [ ]:
SYN_IDX = {c: [vocab_idx[w] for w in VOC_SYNONYMS[c] if w in vocab_idx] for c in VOC_CLASSES}
print({c: len(ix) for c, ix in SYN_IDX.items()})

## Load data

In [ ]:
with h5py.File(PATHS["full"], "r") as f:
    img_ids = f["test_img_ids"][:].ravel()
    full    = f["test_embeddings"][:]
    gt      = f["test_category"][:].astype(bool)

enc, num_obj = {}, {}
for mode in METHODS:
    with h5py.File(PATHS[mode], "r") as f:
        id2row = {iid: r for r, iid in enumerate(f["test_obj_img_ids"][:].ravel())}
        n_all  = f["test_num_obj"][:].ravel()
        group  = f["test_obj_embeddings"]
        objs, ns = [], []
        for iid in tqdm(img_ids, desc=f"load {mode}"):
            r = id2row[iid]
            n = min(int(n_all[r]), NUM_OBJECTS)
            objs.append(group[f"obj_embeddings_{r}"][()][:n])
            ns.append(n)
    enc[mode], num_obj[mode] = objs, np.array(ns)

print(f"{len(img_ids)} test images | mean objects crop {num_obj['crop'].mean():.2f}, circle {num_obj['circle'].mean():.2f} | "
      f"num_obj identical: {(num_obj['crop'] == num_obj['circle']).mean():.4f}")

In [ ]:
from torch.utils.data import DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from data import PascalVOCDataset
from utils import filter_boxes

if os.path.exists(DET_CACHE):
    with open(DET_CACHE, "rb") as f:
        det_labels = pickle.load(f)
    print(f"cache loaded: {DET_CACHE}")
else:
    device     = "cuda" if torch.cuda.is_available() else "cpu"
    coco_names = MaskRCNN_ResNet50_FPN_Weights.COCO_V1.meta["categories"]
    det_model  = maskrcnn_resnet50_fpn(pretrained=True).eval().to(device)   # identical to generate_concepts.py

    voc_test = PascalVOCDataset(f"{ROOT}/data/voc", year="2012", image_set="val", download=False)
    loader   = DataLoader(voc_test, batch_size=32, shuffle=False, num_workers=4)

    det_labels = {}   
    with torch.no_grad():
        for images, _, _, idx in tqdm(loader, desc="Mask R-CNN"):
            images  = images.to(device)
            outputs = det_model(images)
            for img, o, iid in zip(images, outputs, idx):
                h, w = img.shape[1:]
                sel = filter_boxes(o["boxes"], o["scores"], w * h, iou_threshold=IOU_THRESHOLD,
                                   min_size=MIN_OBJ_SIZE, max_size=MAX_OBJ_SIZE, minimum_score=PROPOSAL_SCORE)
                sel    = [p.item() for p in sel]
                labels = o["labels"][sel].cpu()
                scores = o["scores"][sel].cpu()
                labels = labels[scores > MIN_SCORE][:NUM_OBJECTS]
                det_labels[int(iid)] = [coco_names[l] for l in labels.tolist()]

    with open(DET_CACHE, "wb") as f:
        pickle.dump(det_labels, f)
    print(f"cache saved: {DET_CACHE}")

# consistency check against the stored encodings
n_det = np.array([len(det_labels[int(iid)]) for iid in img_ids])
print(f"number of detections == test_num_obj (crop):   {(n_det == num_obj['crop']).mean():.4f}")
print(f"number of detections == test_num_obj (circle): {(n_det == num_obj['circle']).mean():.4f}")
mapped = sum(COCO_TO_VOC.get(n) is not None for l in det_labels.values() for n in l)
print(f"detections with VOC class: {mapped}/{n_det.sum()}")

In [ ]:
rows, sat_rows = [], []
for i, iid in enumerate(tqdm(img_ids, desc="Analysis")):
    dets = [COCO_TO_VOC.get(n) for n in det_labels[int(iid)]]
    for m in METHODS:
        stack = np.vstack([full[i][None], enc[m][i]])   
        count = (stack != 0).sum(0)
        n_src = stack.shape[0]

        if n_src - 1 >= 2:
            active = stack.sum(0) != 0
            if active.any():
                sat_rows.append({"img_id": iid, "method": m, "n_obj": n_src - 1,
                                 "n_active": int(active.sum()),
                                 "frac_all_sources": float((count[active] == n_src).mean())})

        for c in np.flatnonzero(gt[i]):
            cls = VOC_CLASSES[c]
            if not SYN_IDX[cls]:
                continue
            rows.append({"img_id": iid, "cls": cls, "method": m, "n_obj": n_src - 1,
                         "n_det": dets.count(cls), "count_max": int(count[SYN_IDX[cls]].max())})

df  = pd.DataFrame(rows)
sat = pd.DataFrame(sat_rows)
df["n_bin"] = np.where(df["n_det"] >= 4, "≥4", df["n_det"].astype(str))
print(f"{df[df.method == 'crop'].shape[0]} (image, GT class) pairs | {sat[sat.method == 'crop'].shape[0]} images with ≥ 2 objects")

In [ ]:
BINS = ["0", "1", "2", "3", "≥4"]
summary = []
for m in METHODS:
    d = df[df.method == m]
    rho_all, p_all = spearmanr(d["n_det"], d["count_max"])
    d1 = d[d["n_det"] >= 1]
    rho_1, p_1 = spearmanr(d1["n_det"], d1["count_max"])
    row = {"method": m, "rho (all)": rho_all, "p (all)": p_all, "rho (n_det ≥ 1)": rho_1, "p (n_det ≥ 1)": p_1}
    for b in BINS:
        db = d[d["n_bin"] == b]
        row[f"mean count | {b}"] = db["count_max"].mean()
        row[f"N | {b}"]       = len(db)
    summary.append(row)

count_summary = pd.DataFrame(summary).set_index("method")
print("Bin '0' = GT class present but not detected (in addition to 1, 2, 3, ≥4)")
count_summary.T.round(3)

In [ ]:
# Saturation (images with ≥ 2 objects)
sat_summary = sat.groupby("method").agg(
    n_images=("img_id", "size"),
    mean_frac_all_sources=("frac_all_sources", "mean"),
    median_frac_all_sources=("frac_all_sources", "median"),
    mean_active=("n_active", "mean"),
).loc[list(METHODS)]
display(sat_summary.round(4))

# additionally broken down by object count
sat.assign(n_bin=np.where(sat["n_obj"] >= 4, "≥4", sat["n_obj"].astype(str))) \
   .pivot_table(index="n_bin", columns="method", values="frac_all_sources", aggfunc="mean")[list(METHODS)].round(4)

## Ablation: sum, count, binary count 

In [ ]:
def wandb_test_map(run_name):
    """test_map of the latest local wandb run with this name (None if not found)."""
    hits = []
    for d in sorted(glob.glob(f"{ROOT}/wandb/run-*")):
        meta, summ = f"{d}/files/wandb-metadata.json", f"{d}/files/wandb-summary.json"
        if not (os.path.exists(meta) and os.path.exists(summ)):
            continue
        args = json.load(open(meta)).get("args", [])
        if "--wandb_run_name" in args and args[args.index("--wandb_run_name") + 1] == run_name:
            s = json.load(open(summ))
            if "test_map" in s:
                hits.append(s["test_map"])
    return hits[-1] if hits else None

def pct(run_name):
    v = wandb_test_map(run_name)
    return None if v is None else 100 * v

ablation = pd.DataFrame([
    {"method": m,
     "Sum":          pct(f"voc_{m}_sum"),
     "Count":        pct(f"voc_{m}_count_lr001"),
     "Binary Count": pct(f"voc_{m}_presence")}  # runs trained under old name presence
    for m in METHODS
]).set_index("method")
ablation.round(2)